In [1]:
import pytest
import ipytest
from builder import *

In [2]:
ipytest.autoconfig()

In [3]:
%%ipytest

%%ipytest

import pytest

# --- Test Data Setup ---

@pytest.fixture
def valid_design():
    return {
        'environment': {
            'dimensions': [5.0, 4.0, 3.0],
            'wall_resolution': 10,
            'reflectivity': {
                'floor': 0.2,
                'ceiling': 0.7,
                'walls': 0.5
            },
            'special_surfaces': [
                {'type': 'window', 'dims': [1, 1], 'center': [2.5, 4.0, 1.5]},
                {'type': 'RIS', 'dims': [0.5, 0.5], 'center': [0, 2.0, 1.5]},
                {'type': 'window', 'dims': [1, 2], 'center': [5.0, 2.0, 1.5]}
            ]
        }
    }

# --- 1. Parsing Logic Tests ---

def test_room_builder_dimensions(valid_design):
    builder = RoomBuilder(valid_design)
    assert builder.L == 5.0
    assert builder.W == 4.0
    assert builder.H == 3.0
    assert builder.res == (10, 10)

def test_room_builder_reflectivity(valid_design):
    builder = RoomBuilder(valid_design)
    assert builder.floor_refl == 0.2
    assert builder.ceiling_refl == 0.7
    assert builder.wall_refl == 0.5
    assert builder.refl == [0.2, 0.7, 0.5]

def test_special_surface_filtering(valid_design):
    builder = RoomBuilder(valid_design)
    # Based on fixture: 2 windows, 1 RIS
    assert len(builder.windows) == 2
    assert len(builder.RIS) == 1
    assert builder.windows[0]['type'] == 'window'
    assert builder.RIS[0]['type'] == 'RIS'

# --- 2. Error and Edge Case Tests ---

def test_invalid_surface_type(valid_design):
    builder = RoomBuilder(valid_design)
    with pytest.raises(ValueError, match="Invalid surface type"):
        builder.get_surfaces_by_type("door")

def test_missing_special_surfaces():
    # Design without the 'special_surfaces' key
    sparse_design = {
        'environment': {
            'dimensions': [5, 5, 5],
            'wall_resolution': 10,
            'reflectivity': {'floor': 0.1, 'ceiling': 0.1, 'walls': 0.1}
        }
    }
    builder = RoomBuilder(sparse_design)
    assert builder.windows == []
    assert builder.RIS == []

def test_console_output(valid_design, capsys):
    # Tests if the 'console=True' flag actually prints to stdout
    RoomBuilder(valid_design, console=True)
    captured = capsys.readouterr()
    assert "L: 5.0" in captured.out
    assert "Found 2 window surfaces" in captured.out

......                                                                                       [100%]
6 passed in 0.02s
......                                                                                       [100%]
6 passed in 0.01s
